# Lesson 16 Lab — From Kernel Evidence to Inference Engineering

**Puzzle:** What makes a GPU optimization result useful beyond one notebook—and what evidence demonstrates inference-engineering skill?

This notebook retains one complete RTX 5090 execution.


## Why this matters

CUDA operator work and LLM inference engineering overlap but are not identical. Kernel work emphasizes memory access, instruction selection, tiling, occupancy, and correctness. Inference work additionally covers model phases, batching, KV state, APIs, observability, capacity, and release decisions. A strong project connects a bottleneck hypothesis to code, controlled measurement, model/service impact, and a rollback gate.


## 0. Predict before running

1. Predict how many earlier lessons have complete artifacts after a full run.
2. Classify one lesson as hardware model, kernel measurement, or systems decision.
3. Write the decision that your strongest benchmark supports.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The lab audits the first fifteen Chapter 04 artifacts as an evidence portfolio. It counts complete environment identities, evidence labels, metrics, analyses, and conclusions, then maps lessons to hardware, kernel, and inference layers. This is executable quality control rather than a salary forecast or job-market claim. Missing artifacts remain visible as missing instead of being filled with invented results.

- A microbenchmark is stronger when it names the system decision it informs.
- Evidence must preserve environment, workload, comparison, and boundary.
- Inference engineering spans operator, runtime, service, and release layers.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["hardware mechanism"] --> B["kernel hypothesis"]
  B --> C["controlled measurement"]
  C --> D["model/service impact"]
  D --> E["acceptance + rollback decision"]
  E --> F["reproducible portfolio evidence"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 16
LESSON_TITLE = 'From Kernel Evidence to Inference Engineering'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260829
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | a list of lesson titles without machine-readable evidence |
| Candidate | the executed artifacts for Lessons 01–15 |
| Held constant | required artifact fields and fixed lesson-layer mapping |
| Measurements | artifact completeness, evidence-label coverage, and layer coverage |
| Evidence | `compatibility-probe` |

**Experiment:** Audit the preceding structured artifacts and build an evidence-coverage matrix.


## 6. Inspect the code

The code walks sibling lesson directories, validates a minimal schema, counts evidence classes, and prints missing fields. It never edits earlier evidence or treats file existence as proof that a conclusion is correct.

Do not run until the code matches the frozen table.


In [2]:
chapter = Path.cwd().parent
required_top = ("lesson", "title", "environment", "evidence_label", "metrics", "analysis", "conclusion")
layer_map = {
    "hardware-model": set(range(1, 4)) | {7, 9, 10},
    "kernel-measurement": {4, 5, 6, 8, 11, 13, 14, 15},
    "systems-decision": {3, 4, 6, 7, 11, 15},
}
records = []
for lesson in range(1, 16):
    matches = sorted(chapter.glob(f"{lesson:02d}-*/artifacts/rtx5090-result.json"))
    if not matches:
        records.append({"lesson": lesson, "found": False, "complete": False,
                        "missing": list(required_top), "evidence_label": None})
        continue
    data = json.loads(matches[0].read_text(encoding="utf-8"))
    missing = [field for field in required_top if data.get(field) in (None, "", {})]
    env_missing = [field for field in ("gpu", "compute_capability", "torch", "cuda_runtime")
                   if not data.get("environment", {}).get(field)]
    records.append({"lesson": lesson, "found": True, "complete": not missing and not env_missing,
                    "missing": missing, "environment_missing": env_missing,
                    "evidence_label": data.get("evidence_label")})

found = sum(r["found"] for r in records)
complete = sum(r["complete"] for r in records)
labels = sorted({r["evidence_label"] for r in records if r["evidence_label"]})
represented_layers = sorted(name for name, lessons in layer_map.items()
                            if any(r["complete"] and r["lesson"] in lessons for r in records))
metrics = {
    "artifacts_expected": 15, "artifacts_found": found,
    "complete_artifacts": complete, "completion_rate": complete / 15,
    "evidence_labels": labels, "evidence_labels_represented": len(labels),
    "layers": represented_layers, "layers_represented": len(represented_layers),
    "records": records,
}
analysis = (
    f"The portfolio audit found {found}/15 artifacts and {complete}/15 complete records, "
    f"covering {len(labels)} evidence labels and {len(represented_layers)} project layers. "
    "Schema completeness is necessary but does not validate experimental causality."
)
print(json.dumps(metrics, indent=2))


{
  "artifacts_expected": 15,
  "artifacts_found": 15,
  "complete_artifacts": 15,
  "completion_rate": 1.0,
  "evidence_labels": [
    "capacity-model",
    "numerical-model",
    "pytorch-gpu"
  ],
  "evidence_labels_represented": 3,
  "layers": [
    "hardware-model",
    "kernel-measurement",
    "systems-decision"
  ],
  "layers_represented": 3,
  "records": [
    {
      "lesson": 1,
      "found": true,
      "complete": true,
      "missing": [],
      "environment_missing": [],
      "evidence_label": "numerical-model"
    },
    {
      "lesson": 2,
      "found": true,
      "complete": true,
      "missing": [],
      "environment_missing": [],
      "evidence_label": "numerical-model"
    },
    {
      "lesson": 3,
      "found": true,
      "complete": true,
      "missing": [],
      "environment_missing": [],
      "evidence_label": "capacity-model"
    },
    {
      "lesson": 4,
      "found": true,
      "complete": true,
      "missing": [],
      "environment_miss

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Artifacts found | 15 |
| Complete artifacts | 15 |
| Completion rate | 100.00% |
| Evidence labels represented | 3 |
| Portfolio layers represented | 3 |


## 8. Explain rather than overclaim

The portfolio audit found 15/15 artifacts and 15/15 complete records, covering 3 evidence labels and 3 project layers. Schema completeness is necessary but does not validate experimental causality.

**Evidence boundary:** Repository artifacts and installed surfaces were inspected. Schema or API availability is not equivalent to validating an experiment's causal conclusion.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 16, "title": 'From Kernel Evidence to Inference Engineering', "environment": ENV,
    "evidence_label": 'compatibility-probe', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Present performance work as a chain from mechanism to reproducible evidence to a bounded system decision.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 16,
  "title": "From Kernel Evidence to Inference Engineering",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260829
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "artifacts_expected": 15,
    "artifacts_found": 15,
    "complete_artifacts": 15,
    "completion_rate": 1.0,
    "evidence_labels": [
      "capacity-model",
      "numerical-model",
      "pytorch-gpu"
    ],
    "evidence_labels_represented": 3,
    "layers": [
      "hardware-model",
      "kernel-measurement",
      "systems-decision"
    ],
    "layers_represented": 3,
    "records": [
      {
        "lesson": 1,
        "found": true,
        "complete": true,
        "missing": [],
        "environment_missing": [],
        "evidence_label": "numerical-model"
      },
      {
        "lesson": 2,
        "found": true,
        "complete": true,

## 10. Make the decision

> Present performance work as a chain from mechanism to reproducible evidence to a bounded system decision.

**Failure analysis:** Schema completeness cannot detect a flawed experiment or unsupported interpretation. Human review, reproduction, and profiler evidence remain necessary.


## 11. Extend the evidence

Add a review rubric for causal controls, correctness tolerance, profiler trace, end-to-end impact, and rollback rehearsal; score one portfolio item manually.

See [`README.md`](README.md) for the full explanation and references.
